Google Colab code: plots for 
(1) SLM-only comparisons and
(2) Best-SLM vs LLM baselines (gap reduction)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
# Paths
METRICS_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/metrics_outputs"


FSD_CSV = os.path.join(METRICS_DIR, "fsd_metrics_all_models.csv")
FCS_CSV = os.path.join(METRICS_DIR, "fcs_metrics_all_models.csv")
DMA_CSV = os.path.join(METRICS_DIR, "dma_metrics_all_models.csv")

FSD_LAT_CSV = os.path.join(METRICS_DIR, "latency/fsd_metrics_all_models_local.csv")
FCS_LAT_CSV = os.path.join(METRICS_DIR, "latency/fcs_metrics_all_models_local.csv")
DMA_LAT_CSV = os.path.join(METRICS_DIR, "latency/dma_metrics_all_models_local.csv")

# Save figures back to Drive (recommended)
OUT_DIR = os.path.join(METRICS_DIR, "figs")
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# Load metrics + latency for each subject
def load_subject(metrics_csv, lat_csv, subject):
    df = pd.read_csv(metrics_csv)
    lat = pd.read_csv(lat_csv)
    # normalize model column name
    df.columns  = [c.strip() for c in df.columns]
    lat.columns = [c.strip() for c in lat.columns]
    # rename 'model' -> 'Model' if needed
    df  = df.rename(columns={"model": "Model"})
    lat = lat.rename(columns={"model": "Model"})
    # merge latency columns into main df
    df = df.merge(lat, on="Model", how="left")
    df["subject"] = subject
    return df

fsd_df = load_subject(FSD_CSV, FSD_LAT_CSV, "FSD")
fcs_df = load_subject(FCS_CSV, FCS_LAT_CSV, "FCS")
dma_df = load_subject(DMA_CSV, DMA_LAT_CSV, "DMA")

all_df = pd.concat([fsd_df, fcs_df, dma_df], ignore_index=True)
print(all_df.shape)
print(all_df.columns.tolist())
print(all_df["Model"].unique())

(15, 13)
['Model', 'BLEU', 'ROUGE-L', 'METEOR', 'BERTScore_F1', 'latency_mean_s_x', 'latency_median_s_x', 'latency_mean_s_y', 'latency_median_s_y', 'tps_mean', 'tps_median', 'n_tok_mean', 'subject']
['BASE' 'LoRA' 'LoRA+RAG' 'LLM (Llama-3.3-70B)' 'LLM (Qwen3-32B)']


In [6]:

ALL_MODELS_ORDER = [
    "BASE",
    "LoRA",
    "LoRA+RAG",
    # "LLM (Llama-3.3-70B)",
    "LLM (Qwen3-32B)",
]

def plot_all_models(all_df: pd.DataFrame, metric: str, title: str, filename: str, fmt="%.3f"):
    subjects = ["FSD", "FCS", "DMA"]
    models = ALL_MODELS_ORDER

    df = all_df[all_df["Model"].isin(models)].copy()
    df["Model"] = pd.Categorical(df["Model"], categories=models, ordered=True)

    pivot = (
        df.pivot_table(index="subject", columns="Model", values=metric, aggfunc="mean")
          .reindex(subjects)
          .reindex(columns=models)
    )

    x = np.arange(len(subjects))
    width = 0.16
    offsets = np.linspace(-2, 2, len(models)) * width

    fig, ax = plt.subplots(figsize=(11, 5.6))

    bars = []
    for i, m in enumerate(models):
        b = ax.bar(x + offsets[i], pivot[m].values, width=width, label=m)
        bars.append(b)

    ax.set_xticks(x)
    ax.set_xticklabels(subjects)
    ax.set_ylabel(metric)
    ax.set_title(title)

    for b in bars:
        ax.bar_label(b, fmt=fmt, padding=3, fontsize=9)

    vals = pivot.values.flatten()
    vals = vals[~np.isnan(vals)]
    if len(vals) > 0:
        vmin, vmax = float(np.min(vals)), float(np.max(vals))
        if vmax == vmin:
            ax.set_ylim(vmin - 1, vmax + 1)
        else:
            top_pad = 0.18
            bottom_pad = 0.06
            y0 = vmin - abs(vmax - vmin) * bottom_pad
            y1 = vmax + abs(vmax - vmin) * top_pad
            ax.set_ylim(y0, y1)

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)

    fig.tight_layout()
    out_path = os.path.join(OUT_DIR, filename)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)


# Calls
# plot_all_models(all_df, "BLEU",              "BLEU-2 (all models)",              "all_bleu.png",            fmt="%.3f")
# plot_all_models(all_df, "ROUGE-L",           "ROUGE-L (all models)",             "all_rougeL.png",          fmt="%.3f")
# plot_all_models(all_df, "METEOR",            "METEOR (all models)",              "all_meteor.png",          fmt="%.3f")
# plot_all_models(all_df, "BERTScore_F1",      "BERTScore F1 (all models)",        "all_bertscore.png",       fmt="%.3f")
plot_all_models(all_df, "latency_mean_s",    "Latency mean (s)",    "all_latency_mean.png",    fmt="%.2f")
plot_all_models(all_df, "latency_median_s",  "Latency median (s)",  "all_latency_median.png",  fmt="%.2f")
plot_all_models(all_df, "tps_mean",          "TPS mean",            "all_tps_mean.png",        fmt="%.2f")
plot_all_models(all_df, "tps_median",        "TPS median",          "all_tps_median.png",      fmt="%.2f")
plot_all_models(all_df, "n_tok_mean",        "Tokens mean",         "all_n_tok_mean.png",      fmt="%.1f")

KeyError: 'latency_mean_s'